# Notebook 02 - Analisis Fiscal Argentina 2020-2026

**Fuente:** Secretaria de Hacienda | Sector Publico Base Caja (AIF) + IMIG  
**Datos en:** https://github.com/santiagoriverti/cuentas_publicas  
**Unidad:** millones de pesos corrientes

### Indice
1. Resultado Primario y Financiero 2020-2026
2. Composicion de Ingresos
3. Composicion del Gasto
4. Transferencias a Provincias
5. Desagregacion por Subsector Institucional
6. Resumen del ajuste 2024-2026

In [ ]:
# Celda 1 - Dependencias
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q pandas matplotlib

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (15, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
print('OK')

In [ ]:
# Celda 2 - Carga de datos desde GitHub
REPO = 'https://raw.githubusercontent.com/santiagoriverti/cuentas_publicas/main'

df_aif  = pd.read_csv(f'{REPO}/output/aif_consolidado.csv',  parse_dates=['fecha'])
df_imig = pd.read_csv(f'{REPO}/output/imig_consolidado.csv', parse_dates=['fecha'])

# Solo mensual para el analisis principal
df = df_aif[df_aif['periodo'] == 'mensual'].copy()

print(f'AIF mensual: {len(df):,} registros | {df["fecha"].min().strftime("%Y-%m")} a {df["fecha"].max().strftime("%Y-%m")}')
df.head(2)

In [ ]:
# Helper: extraer serie de un concepto y subsector
def get_serie(concepto, subsector='total_adm_nacional'):
    mask = (df['concepto_codigo'] == concepto) & (df['subsector'] == subsector)
    return df[mask].set_index('fecha')['valor_millones_pesos'].sort_index()

def fmt_miles_mm(x, _):
    return f'{x/1000:,.0f}'

def fmt_mm(x, _):
    return f'{x:,.0f}'

## 1. Resultado Primario y Financiero 2020-2026

In [ ]:
primario   = get_serie('XIV_RESULTADO_PRIMARIO')
financiero = get_serie('XV_RESULTADO_FINANCIERO')

fig, ax = plt.subplots(figsize=(15, 6))
colors = ['#27ae60' if v >= 0 else '#e74c3c' for v in primario.values]
ax.bar(primario.index, primario.values / 1000, width=25, color=colors, alpha=0.85, label='Resultado Primario')
ax.plot(financiero.index, financiero.values / 1000, 'o-', color='#2c3e50',
        linewidth=2, markersize=3, label='Resultado Financiero')
ax.axhline(0, color='black', linewidth=0.8)
ax.yaxis.set_major_formatter(mtick.FuncFormatter(fmt_miles_mm))
ax.set_title('Resultado Primario y Financiero - Administracion Nacional mensual (miles de MM ARS)', fontsize=13)
ax.set_ylabel('Miles de MM ARS')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# Tabla anual
anual = pd.DataFrame({'Primario': primario, 'Financiero': financiero})
anual.index = pd.to_datetime(anual.index)
tabla = (anual.groupby(anual.index.year).sum() / 1_000_000).round(2)
tabla.columns = ['Result. Primario (billones ARS)', 'Result. Financiero (billones ARS)']
print('\nResumen anual:')
print(tabla.to_string())

## 2. Composicion de Ingresos

In [ ]:
ing_total  = get_serie('I_INGRESOS_CORRIENTES')
ing_trib   = get_serie('I1_INGRESOS_TRIBUTARIOS')
ing_ss     = get_serie('I2_APORTES_SEG_SOCIAL')

fig, ax = plt.subplots(figsize=(15, 5))
ax.stackplot(ing_trib.index,
             ing_trib.values / 1000,
             ing_ss.reindex(ing_trib.index).fillna(0).values / 1000,
             labels=['Ingresos Tributarios', 'Aportes Seg. Social'],
             colors=['#2ecc71', '#3498db'], alpha=0.8)
ax.plot(ing_total.index, ing_total.values / 1000, 'k-', linewidth=1.5, label='Total Ingresos Corrientes')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(fmt_miles_mm))
ax.set_title('Ingresos Corrientes - Administracion Nacional (miles de MM ARS)', fontsize=13)
ax.set_ylabel('Miles de MM ARS')
ax.legend(loc='upper left')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Composicion del Gasto

In [ ]:
gasto_total  = get_serie('II_GASTOS_CORRIENTES')
prestaciones = get_serie('II3_PRESTACIONES_SEG_SOCIAL')
intereses    = get_serie('II2_INTERESES')
transferencias = get_serie('II4_TRANSF_CORRIENTES_GASTO')

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Gastos corrientes totales
axes[0].bar(gasto_total.index, gasto_total.values / 1000, width=25, color='#e74c3c', alpha=0.7)
axes[0].set_title('Gastos Corrientes Totales')
axes[0].yaxis.set_major_formatter(mtick.FuncFormatter(fmt_miles_mm))
axes[0].set_ylabel('Miles de MM ARS')
axes[0].grid(axis='y', alpha=0.3)

# Intereses
axes[1].bar(intereses.index, intereses.values / 1000, width=25, color='#8e44ad', alpha=0.7)
axes[1].set_title('Intereses de Deuda')
axes[1].yaxis.set_major_formatter(mtick.FuncFormatter(fmt_miles_mm))
axes[1].set_ylabel('Miles de MM ARS')
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('Composicion del Gasto - Administracion Nacional mensual', fontsize=13)
plt.tight_layout()
plt.show()

# Prestaciones
if len(prestaciones) > 0:
    fig, ax = plt.subplots(figsize=(15, 4))
    ax.bar(prestaciones.index, prestaciones.values / 1000, width=25, color='#f39c12', alpha=0.8)
    ax.set_title('Prestaciones de la Seguridad Social (jubilaciones, pensiones, etc.)', fontsize=12)
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(fmt_miles_mm))
    ax.set_ylabel('Miles de MM ARS')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

## 4. Transferencias a Provincias y CABA

In [ ]:
transf_corr = df[
    (df['concepto_codigo'] == 'II4b1_TRANSF_PROVINCIAS_CABA') &
    (df['subsector'] == 'tesoro_nacional')
].set_index('fecha')['valor_millones_pesos'].sort_index()

transf_cap = df[
    (df['concepto_codigo'] == 'V2a_TRANSF_CAPITAL_PROVINCIAS') &
    (df['subsector'] == 'tesoro_nacional')
].set_index('fecha')['valor_millones_pesos'].sort_index()

fig, ax = plt.subplots(figsize=(15, 5))
ax.bar(transf_corr.index, transf_corr.values / 1000, width=25,
       label='Transferencias Corrientes', color='#9b59b6', alpha=0.8)
if len(transf_cap) > 0:
    bottom = transf_corr.reindex(transf_cap.index).fillna(0)
    ax.bar(transf_cap.index, transf_cap.values / 1000, width=25,
           bottom=bottom.values / 1000,
           label='Transferencias de Capital', color='#e67e22', alpha=0.8)

ax.set_title('Transferencias del Tesoro a Provincias y CABA (mensual)', fontsize=13)
ax.set_ylabel('Miles de MM ARS')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(fmt_miles_mm))
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# Comparacion anual
df_prov = pd.DataFrame({'Corrientes': transf_corr, 'Capital': transf_cap})
df_prov.index = pd.to_datetime(df_prov.index)
tabla_prov = (df_prov.groupby(df_prov.index.year).sum() / 1000).round(1)
tabla_prov.columns = ['Transf. Corrientes (miles MM ARS)', 'Transf. Capital (miles MM ARS)']
print('Transferencias anuales Tesoro -> Provincias/CABA:')
print(tabla_prov.to_string())

## 5. Desagregacion por Subsector Institucional

In [ ]:
subsectores = {
    'tesoro_nacional':    'Tesoro Nacional',
    'inst_seg_social':    'Seg. Social (ANSES)',
    'org_descentralizados': 'Org. Descentralizados',
    'rec_afectados':      'Rec. Afectados',
}
colors_ss = ['#e74c3c', '#2ecc71', '#3498db', '#f39c12']

fig, ax = plt.subplots(figsize=(15, 6))
for (ss, label), color in zip(subsectores.items(), colors_ss):
    serie = df[
        (df['concepto_codigo'] == 'XIV_RESULTADO_PRIMARIO') &
        (df['subsector'] == ss)
    ].set_index('fecha')['valor_millones_pesos'].sort_index()
    if len(serie) > 0:
        ax.plot(serie.index, serie.values / 1000, label=label, linewidth=1.8, color=color)

ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_title('Resultado Primario por Subsector Institucional (mensual)', fontsize=13)
ax.set_ylabel('Miles de MM ARS')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(fmt_miles_mm))
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. El ajuste fiscal 2024-2026 en numeros

In [ ]:
# Comparacion acumulada por año
conceptos = {
    'I_INGRESOS_CORRIENTES':       'Ingresos corrientes',
    'II_GASTOS_CORRIENTES':        'Gastos corrientes',
    'V_GASTOS_CAPITAL':            'Gastos de capital',
    'II2_INTERESES':               'Intereses',
    'XIV_RESULTADO_PRIMARIO':      'Resultado PRIMARIO',
    'XV_RESULTADO_FINANCIERO':     'Resultado FINANCIERO',
}

rows = []
for cod, nombre in conceptos.items():
    serie = df[
        (df['concepto_codigo'] == cod) &
        (df['subsector'] == 'total_adm_nacional')
    ].copy()
    serie['anio'] = serie['fecha'].dt.year
    anual = serie.groupby('anio')['valor_millones_pesos'].sum()
    for yr in [2022, 2023, 2024, 2025]:
        if yr in anual.index:
            rows.append({'Concepto': nombre, 'Año': yr, 'Valor (MM ARS)': anual[yr]})

resumen = pd.DataFrame(rows)
pivot_res = resumen.pivot(index='Concepto', columns='Año', values='Valor (MM ARS)') / 1_000_000
print('Resumen fiscal anual (en billones de ARS):')
print(pivot_res.round(1).to_string())